# Sionna 0.19 – Differentiable Ray Tracing: Material Calibration & TX Optimization
**Kernel:** `sionna019` · Python 3.10 · Sionna 0.19.2 · TensorFlow 2.15

Fixed from original sionna_019_differentiable_rt.ipynb:
- `load_scene()` — removed invalid `merge_shapes=True` argument (not supported in Sionna 0.19)
- `UTM_EPSG` — corrected from 32631 (Paris/zone-31N) to 32630 (UK/zone-30N) for London scenes
- DEM CRS auto-detection (BNG vs WGS84) for correct coordinate lookup

Implements NVlabs/diff-rt pattern: NMSE OFDM loss · Adam · `check_mat()` · TX orientation optimization.

## CELL 0 · Environment Setup & Imports

In [ ]:
import os, sys, json, csv, time, warnings, importlib
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from pyproj import Transformer

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver

# ── Mitsuba (ray casting) ──────────────────────────────────────────────────────
_HAS_MI = False
try:
    import mitsuba as mi
    try:   mi.set_variant('cuda_ad_mono_polarized')
    except: mi.set_variant('llvm_ad_mono_polarized')
    _HAS_MI = True
    print(f'Mitsuba : {mi.variant()}')
except ImportError:
    print('Mitsuba : NOT available – ray-cast ground height disabled')

# ── rasterio (DEM lookup) ──────────────────────────────────────────────────────
_HAS_RIO = False
try:
    import rasterio as rio
    _HAS_RIO = True
except ImportError:
    print('rasterio: NOT available – DEM elevation disabled')

# ── OFDM helpers (diff-rt NMSE loss) ──────────────────────────────────────────
_HAS_OFDM = False
for _pkg in ('sionna.channel', 'sionna.channel.ofdm'):
    try:
        _m = importlib.import_module(_pkg)
        cir_to_ofdm_channel    = _m.cir_to_ofdm_channel
        subcarrier_frequencies = _m.subcarrier_frequencies
        _HAS_OFDM = True
        print(f'OFDM    : OK  ({_pkg})')
        break
    except (ImportError, AttributeError):
        continue
if not _HAS_OFDM:
    print('OFDM    : NOT found – power-domain fallback will be used')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')
print(f'GPU(s)  : {[g.name for g in tf.config.list_physical_devices("GPU")]}')

# ── Shared helpers ─────────────────────────────────────────────────────────────
def _safe(v):
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')

## CELL 1 · Project & Scene Detection

Auto-detects scene.xml from the project folder.
Reads scene bounds from XML `<default>` tags.

In [ ]:
# ── Nottingham 11km scene (11km × 10km, 3602.5 MHz Ofcom site) ───────────────
CITY_NAME    = 'Nottingham'
SCENE_VERSION = '11km'

_suffix = f'_{SCENE_VERSION}' if SCENE_VERSION != 'default' else ''
BASE_DIR = os.path.expanduser(f'~/Documents/FYP2026/{CITY_NAME.lower()}{_suffix}')
PROJECT_PATH = BASE_DIR
SCENE_XML    = os.path.join(BASE_DIR, 'scene', 'scene.xml')
DEM_TIFF     = os.path.join(BASE_DIR, 'scene', 'dem_wgs84.tif')

# Nottingham 11km bbox
WEST, EAST   = -1.409134, -1.245094
SOUTH, NORTH =  52.910276, 53.009090

XML_OK = os.path.exists(SCENE_XML)
print(f'Project   : {CITY_NAME} {SCENE_VERSION}')
print(f'Base dir  : {BASE_DIR}')
print(f'Scene XML : {SCENE_XML}  {"✓" if XML_OK else "✗ NOT FOUND"}')
print(f'DEM       : {DEM_TIFF}  {"✓" if os.path.exists(DEM_TIFF) else "✗ NOT FOUND"}')
print(f'Bbox      : lon [{WEST}, {EAST}]  lat [{SOUTH}, {NORTH}]')

# Read scene XML to confirm version
if XML_OK:
    try:
        import xml.etree.ElementTree as _ET
        _root = _ET.parse(SCENE_XML).getroot()
        _ver  = _root.get('version', 'unknown')
        _maj  = int(_ver.split('.')[0]) if _ver and _ver[0].isdigit() else 0
        print(f'XML version : {_ver}  {"✓ Mitsuba 3 compatible" if _maj >= 3 else "✗ needs migration to 3.0.0"}')
    except Exception as _e:
        print(f'XML parse warning: {_e}')

# Output directory
OUTPUT_DIR = os.path.join(BASE_DIR, 'results', 'diff_rt')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

## CELL 2 · Global Configuration

In [ ]:
# ── Ofcom site parameters (Nottingham 3602.5 MHz) ────────────────────────────
# Aligned with sionna019_main_simulation.ipynb — no bias, no site correction.
# Formula: RSSI = TX_CONDUCTED_DBM + 10*log10(sum|a|^2) + RX_EXTRA_GAIN_DB
# Sionna paths.a already includes TX+RX antenna patterns, so only supply
# conducted power and the passive RX chain gain (LNA - losses).
FREQUENCY_HZ    = 3602.5e6      # Ofcom measured frequency
TX_HEIGHT_M     = 17.0          # TX antenna height AGL (m)
TX_POWER_DBM    = 47.8          # amplifier output power (dBm)
TX_CABLE_LOSS   = 2.8           # feeder/cable loss (dB)
TX_GAIN_DBI     = 2.8           # antenna gain (dBi) — handled by Sionna pattern
TX_CONDUCTED_DBM = TX_POWER_DBM - TX_CABLE_LOSS   # 47.8 - 2.8 = 45.0 dBm
EIRP_DBM        = TX_CONDUCTED_DBM   # alias — no +6.2 site correction

# RX chain (Ofcom spec — antenna gain excluded: Sionna handles it via pattern)
RX_AGL_M        = 1.5
RX_GAIN_DBI     = -2.0          # handled by Sionna RX antenna pattern
RX_CABLE_LOSS   = 3.8
RX_SPLITTER_LOSS= 0.0
RX_LNA_GAIN     = 23.3
RX_BPF_LOSS     = 1.5
RX_EXTRA_GAIN_DB = RX_LNA_GAIN - RX_CABLE_LOSS - RX_BPF_LOSS
# = 23.3 - 3.8 - 1.5 = 18.0 dB  (LNA gain minus passive losses only)
SYS_GAIN        = RX_EXTRA_GAIN_DB   # alias used by paths_to_rssi()
SITE_CORRECTION_DB = 0.0             # no manual correction

# TX GPS position (from Ofcom CSV row 01-02)
TX_LAT          = 52.9863
TX_LON          = -1.2559

BANDWIDTH_HZ    = 20e6
NOISE_FLOOR     = -109.0    # Ofcom spec: system noise floor (dBm)

# ── Coordinate system ─────────────────────────────────────────────────────────
UTM_EPSG        = 32630   # UTM zone 30N — covers UK/Nottingham

# ── Input / output CSVs ───────────────────────────────────────────────────────
RX_CSV          = os.path.join(BASE_DIR, 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(BASE_DIR, 'measurements_with_pathloss.csv')
TX_CSV          = os.path.join(BASE_DIR, 'transmitter_positions.csv')

# ── OFDM parameters ───────────────────────────────────────────────────────────
NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if _HAS_OFDM:
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

# ── Path solver parameters ────────────────────────────────────────────────────
MAX_DEPTH      = 15
NUM_SAMPLES_CM = 100_000_000
NUM_SAMPLES_PS = 2_000_000
GRID_SIZE_M    = 5.0

# ── Differentiable RT calibration ─────────────────────────────────────────────
CALIB_STEPS    = 500      # official paper: 10000; 500 is practical for RSSI-only
CALIB_LR       = 5e-3     # Adam learning rate
CALIB_N_RX     = 200      # calibration receivers (stratified by distance)
CALIB_BATCH    = 50       # mini-batch size per gradient step
CALIB_NUM_SAMP = 500_000  # num_samples for compute_paths() per step
CALIB_DEPTH    = 10       # max_depth during calibration — match MAX_DEPTH in main notebook

# ── TX orientation optimization ───────────────────────────────────────────────
ORI_STEPS    = 50
ORI_LR       = 0.01
ORI_NUM_SAMP = 1_000_000

_tx_w    = 10**((EIRP_DBM  - 30) / 10)
_noise_w = 10**((NOISE_FLOOR - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

print('=' * 65)
print('SYSTEM CONFIGURATION — Nottingham 3602.5 MHz (Ofcom)')
print('=' * 65)
print(f'Frequency   : {FREQUENCY_HZ/1e6:.1f} MHz')
print(f'TX EIRP     : {EIRP_DBM} dBm  (pos: {TX_LAT}, {TX_LON}, h={TX_HEIGHT_M}m)')
print(f'SYS_GAIN    : {SYS_GAIN:.1f} dB  (RX net: {RX_GAIN_DBI}+{RX_LNA_GAIN}-{RX_CABLE_LOSS}-{RX_SPLITTER_LOSS}-{RX_BPF_LOSS})')
print(f'SNR scale   : {SNR_SCALE:.2e}')
print(f'UTM EPSG    : {UTM_EPSG}')
print(f'Max depth   : {MAX_DEPTH}')
print(f'RX CSV      : {RX_CSV}  {"✓" if os.path.exists(RX_CSV) else "✗  (run CELL 6c in main notebook first)"}')
print(f'Meas CSV    : {MEASUREMENT_CSV}  {"✓" if os.path.exists(MEASUREMENT_CSV) else "✗"}')
print(f'Calib       : {CALIB_STEPS} steps  LR={CALIB_LR}  N={CALIB_N_RX}  batch={CALIB_BATCH}')
print('=' * 65)

## CELL 3 · Coordinate Utilities + DEM Elevation

- `gps_to_local(lon, lat)` → UTM → subtract scene origin → local XY
- `local_to_gps(x, y)` → reverse
- `get_dem_elevation(local_x, local_y)` → rasterio bilinear on `dem_wgs84.tif`
- `ray_cast_ground_z(x, y)` → Mitsuba ray intersect for terrain height

In [ ]:
# Derive scene center from bbox
center_lon = (WEST + EAST)   / 2
center_lat = (SOUTH + NORTH) / 2

gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    """GPS (lon, lat) to Sionna local XY (metres from scene origin)."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Sionna local XY to GPS (lon, lat)."""
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

# -- DEM bilinear lookup -----------------------------------------------------
dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())
if _is_bng_dem:
    utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)

def get_dem_elevation(local_x, local_y):
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    if _is_bng_dem:
        px, py = utm_to_bng.transform(utm_x, utm_y)
    else:
        px, py = utm_to_gps.transform(utm_x, utm_y)  # WGS84 lon/lat
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if not _HAS_MI: return get_dem_elevation(x, y)
    try:
        ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                       mi.Vector3f(0.0, 0.0, -1.0))
        si = scene.mi_scene.ray_intersect(ray)
        if si.is_valid():
            z_val = si.p.z
            return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
    except Exception:
        pass
    return get_dem_elevation(x, y)

def _scene_bbox():
    """Return (xmin, xmax, ymin, ymax) in local metres. Works Sionna 0.19 and 2.0."""
    for _attr in ('mi_scene', '_scene'):
        try:
            bb = getattr(scene, _attr).bbox()
            return float(bb.min[0]), float(bb.max[0]), float(bb.min[1]), float(bb.max[1])
        except Exception:
            continue
    # Fallback: compute from lon/lat bbox
    wx, sy = gps_to_utm.transform(WEST,  SOUTH)
    ex, ny = gps_to_utm.transform(EAST,  NORTH)
    return (wx - utm_center_x, ex - utm_center_x,
            sy - utm_center_y, ny - utm_center_y)

print('Coordinate utilities ready.')
print(f'  center: ({center_lon:.4f}, {center_lat:.4f})')

## CELL 4 · Load 3-D Scene & Configure Antennas

**FIX:** Removed `merge_shapes=True` — this argument does not exist in Sionna 0.19's `load_scene()`.
The original error was: `TypeError: load_scene() got an unexpected keyword argument 'merge_shapes'`

In [ ]:
if not XML_OK:
    raise RuntimeError(
        'scene.xml not found. Complete Steps 1-3 in the sionna_web UI:\n'
        '  1. Draw area on map\n'
        '  2. Configure materials\n'
        '  3. Click "Generate 3-D Scene"')

print(f'Loading scene from {SCENE_XML} ...')
# Sionna 2.0: merge_shapes=False keeps named objects for material assignment
try:
    scene = load_scene(SCENE_XML, merge_shapes=False)
except TypeError:
    scene = load_scene(SCENE_XML)   # Sionna 0.19 fallback -- no merge_shapes param
scene.frequency = FREQUENCY_HZ

def _make_array(cfg):
    return PlanarArray(
        num_rows           = cfg.get('num_rows',           1),
        num_cols           = cfg.get('num_cols',           1),
        vertical_spacing   = cfg.get('vertical_spacing',   0.5),
        horizontal_spacing = cfg.get('horizontal_spacing', 0.5),
        pattern            = cfg.get('pattern',            'iso'),
        polarization       = cfg.get('polarization',       'V'),
    )

_ant = {}
scene.tx_array = _make_array(_ant.get('tx_array', {}))
scene.rx_array = _make_array(_ant.get('rx_array', {}))

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')
print(f'TX array      : {scene.tx_array}')

# Scene bounding box (uses _scene_bbox() defined in CELL 3)
try:
    _xmin, _xmax, _ymin, _ymax = _scene_bbox()
    print(f'Scene bbox    : X=[{_xmin:.0f}, {_xmax:.0f}]  Y=[{_ymin:.0f}, {_ymax:.0f}]')
except Exception as _be:
    print(f'Scene bbox    : {_be}')


## CELL 4A · Assign ITU-R P.2040-2 Material Properties

In [ ]:
_ITU_DB = {
    'concrete'          : (5.24,  0.130, 0.40, 0.20),
    'brick'             : (3.91,  0.024, 0.30, 0.20),
    'wood'              : (1.99,  0.005, 0.25, 0.30),
    'glass'             : (6.27,  0.012, 0.08, 0.10),
    'metal'             : (1.00,  1e7,   0.05, 0.10),
    'asphalt'           : (3.00,  0.010, 0.35, 0.20),
    'vegetation'        : (1.30,  0.001, 0.75, 0.05),
    'water'             : (81.0,  0.500, 0.02, 0.05),
    'wet_ground'        : (30.0,  0.150, 0.20, 0.20),
    'medium_dry_ground' : (15.0,  0.035, 0.18, 0.20),
    'very_dry_ground'   : (3.00,  0.001, 0.12, 0.20),
    'marble'            : (7.07,  0.020, 0.08, 0.10),
    'plasterboard'      : (2.73,  0.010, 0.12, 0.20),
    'chipboard'         : (2.58,  0.012, 0.14, 0.20),
    'plywood'           : (2.71,  0.014, 0.15, 0.20),
    'ceiling_board'     : (1.50,  0.006, 0.13, 0.20),
    'floorboard'        : (2.00,  0.010, 0.16, 0.20),
}
_DEFAULT_MAT = (4.0, 0.08, 0.30, 0.15)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    for key in _ITU_DB:
        if key in n: return key
    for key in _ITU_DB:
        if any(part in n for part in key.split('_')): return key
    return None

_SCATTER_CFG = {
    'concrete': (0.15, 0.30), 'brick': (0.10, 0.25), 'wood': (0.10, 0.10),
    'glass': (0.05, 0.01), 'metal': (0.05, 0.001), 'wet_ground': (0.05, 0.10),
    'medium_dry_ground': (0.05, 0.10), 'very_dry_ground': (0.05, 0.10),
    'marble': (0.05, 0.30), 'plasterboard': (0.10, 0.02), 'chipboard': (0.10, 0.02),
    'plywood': (0.10, 0.02), 'ceiling_board': (0.10, 0.02), 'floorboard': (0.10, 0.02),
    'vegetation': (0.75, 1.00), 'asphalt': (0.35, 0.15), 'water': (0.02, 0.10),
}

print('=' * 70)
print('ASSIGNING ITU-R MATERIAL PROPERTIES  (auto-match by name)')
print('=' * 70)
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    eps_r, sigma, S, xpd = _ITU_DB.get(key, _DEFAULT_MAT)
    sc, th = _SCATTER_CFG.get(key, (0.20, 0.10))
    try: mat.relative_permittivity = eps_r
    except Exception: pass
    try: mat.conductivity = sigma
    except Exception: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, sc); break
            except: pass
    for a_ in ('xpd_coefficient', 'xpd_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, xpd); break
            except: pass
    try: mat.thickness = th
    except: pass
    print(f'  {mat_name:<32}  matched={key or "DEFAULT":<20}  '
          f'eps={eps_r:.2f}  σ={sigma:.4g}  S={sc:.2f}')
print('Done.')

## CELL 6 · Load Transmitter (GPS → local XY, ray-cast Z)

In [ ]:
print('=' * 70)
print('CELL 6 – LOAD TRANSMITTER')
print('=' * 70)

for nm in list(scene.transmitters.keys()):
    scene.remove(nm)

if os.path.exists(TX_CSV):
    df_tx    = pd.read_csv(TX_CSV)
    row      = df_tx.iloc[0]
    tx_name  = str(row.get('name', 'tx_0'))
    tx_lon   = float(row['lon'])
    tx_lat   = float(row['lat'])
    tx_agl   = float(row.get('height', 25.0))
    tx_power = float(row.get('power_dbm', TX_CONDUCTED_DBM))
    source   = 'CSV'
else:
    _tx_cfg  = _ant.get('transmitters', [{'name':'tx0','position':[0,0,25]}])[0]
    _pos     = _tx_cfg.get('position', [0, 0, 25])
    tx_name  = _tx_cfg.get('name', 'tx0')
    tx_lon, tx_lat = local_to_gps(_pos[0], _pos[1])
    tx_agl   = _pos[2]
    tx_power = TX_CONDUCTED_DBM
    source   = 'project.json'
    print(f'  TX CSV not found – using {source}')

print(f'[1] Source      : {source}')
print(f'    GPS         : ({tx_lon:.6f}, {tx_lat:.6f})  AGL={tx_agl:.1f} m')

local_x, local_y, _ = gps_to_local(tx_lon, tx_lat)
print(f'[2] Local XY    : ({local_x:.2f}, {local_y:.2f})')

ground_z = ray_cast_ground_z(local_x, local_y)
if ground_z == 0.0:
    try:
        _bbox    = scene.mi_scene.bbox()
        ground_z = float(_bbox.min[2])
    except: pass
abs_z = ground_z + tx_agl
print(f'[3] Ground Z    : {ground_z:.2f} m  +  AGL {tx_agl:.1f} m  →  abs Z={abs_z:.2f} m')

tx = Transmitter(name=tx_name,
                 position=(float(local_x), float(local_y), float(abs_z)),
                 power_dbm=float(tx_power))
scene.add(tx)
print(f'[4] ✓ Added TX "{tx_name}"  pos=({local_x:.1f}, {local_y:.1f}, {abs_z:.1f})  EIRP={tx_power:.1f} dBm')

## CELL 7 · Load Receivers (GPS → local XY, ray-cast Z)

In [ ]:
print('=' * 70)
print('CELL 7 – LOAD RECEIVERS')
print('=' * 70)

for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []

if os.path.exists(RX_CSV):
    df_rx = pd.read_csv(RX_CSV)
    print(f'[1] Loaded {len(df_rx)} receivers from {RX_CSV}')
    print('[2] Converting GPS → local XY + ray-cast ground Z ...')
    t0 = time.time()
    for i, row in df_rx.iterrows():
        lon  = float(row['lon'])
        lat  = float(row['lat'])
        agl  = float(row.get('height', 1.5))
        x, y, _ = gps_to_local(lon, lat)
        gz   = ray_cast_ground_z(x, y)
        z    = gz + agl
        nm   = str(row.get('name', f'RX_{i+1:04d}'))
        rx   = Receiver(name=nm, position=(float(x), float(y), float(z)))
        scene.add(rx)
        receivers.append(rx)
    print(f'    Done in {time.time()-t0:.2f} s')
else:
    print('  RX CSV not found – using project.json receivers')
    for rx_cfg in _ant.get('receivers', [{'name':'rx0','position':[100,0,1.5]}]):
        pos = rx_cfg.get('position', [100, 0, 1.5])
        nm  = rx_cfg.get('name', f'rx{len(receivers)}')
        rx  = Receiver(name=nm, position=(float(pos[0]), float(pos[1]), float(pos[2])))
        scene.add(rx)
        receivers.append(rx)

print(f'[3] {len(receivers)} receivers placed')
print('[4] First 5 receivers:')
for rx in receivers[:5]:
    x, y, z = _safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name:<15} XY=({x:8.1f}, {y:8.1f})  Z={z:.2f}  GPS=({lon:.5f}, {lat:.5f})')

try:
    _bbox = scene.mi_scene.bbox()
    ok_x  = all(float(_bbox.min[0]) <= _safe(r.position[0]) <= float(_bbox.max[0]) for r in receivers)
    ok_y  = all(float(_bbox.min[1]) <= _safe(r.position[1]) <= float(_bbox.max[1]) for r in receivers)
    print(f'[5] All inside scene bbox: X={ok_x}  Y={ok_y}')
except: pass

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')

# Load Ofcom measurements for calibration
df_meas = None
rssi_measured_all = None
if os.path.exists(MEASUREMENT_CSV):
    df_meas = pd.read_csv(MEASUREMENT_CSV)
    _rssi_col = next((c for c in df_meas.columns
                      if any(k in c.lower() for k in ['rssi','measurement','dbm','signal'])), None)
    if _rssi_col:
        rssi_measured_all = df_meas[_rssi_col].values.astype(np.float32)
        print(f'Ofcom RSSI loaded : {len(rssi_measured_all)} samples  '
              f'range {rssi_measured_all.min():.1f}\u2013{rssi_measured_all.max():.1f} dBm')
    else:
        print(f'WARNING: no RSSI column in {MEASUREMENT_CSV}')
else:
    print(f'WARNING: {MEASUREMENT_CSV} not found \u2014 run CELL 6c in main notebook first')


## CELL 8 · Pre-Calibration Coverage Map

In [ ]:
print('Pre-calibration coverage map (ITU material defaults) ...')

try:
    _bbox = scene.mi_scene.bbox()
    cx = (float(_bbox.min[0]) + float(_bbox.max[0])) / 2
    cy = (float(_bbox.min[1]) + float(_bbox.max[1])) / 2
except Exception:
    cx = cy = 0.0

ground_z_centre = ray_cast_ground_z(cx, cy)
if ground_z_centre == 0.0:
    ground_z_centre = get_dem_elevation(cx, cy)
cm_height = ground_z_centre + 1.5
print(f'  Coverage map plane Z = {ground_z_centre:.2f} + 1.5 = {cm_height:.2f} m')

cm_pre = scene.coverage_map(
    cm_cell_size        = GRID_SIZE_M,
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_CM,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = False,
)
cm_pre_np = _cm_to_numpy(cm_pre)

pg_db = 10 * np.log10(cm_pre_np[0] + 1e-30)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(pg_db, origin='lower', cmap='jet',
               vmin=np.nanpercentile(pg_db, 5), vmax=np.nanpercentile(pg_db, 99))
plt.colorbar(im, ax=ax, label='Path Gain (dB)')
ax.set_title('Pre-Calibration Coverage Map – ITU Material Defaults')
ax.set_xlabel('X cells'); ax.set_ylabel('Y cells')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_pre_calibration.png'), dpi=150)
plt.show()

# ── Interpolate to RX positions via KDTree ────────────────────────────────────
H, W = pg_db.shape
try:
    _bbox = scene.mi_scene.bbox()
    _gx_min, _gx_max = float(_bbox.min[0]), float(_bbox.max[0])
    _gy_min, _gy_max = float(_bbox.min[1]), float(_bbox.max[1])
except Exception:
    _gx_min = _gy_min = -500.0; _gx_max = _gy_max = 500.0

x_centers = np.linspace(_gx_min, _gx_max, W)
y_centers  = np.linspace(_gy_min, _gy_max, H)
XX, YY = np.meshgrid(x_centers, y_centers)
tree   = KDTree(np.column_stack([XX.ravel(), YY.ravel()]))
rx_coords   = np.array([(_safe(rx.position[0]), _safe(rx.position[1])) for rx in receivers])
_, _indices = tree.query(rx_coords)
pg_at_rx_pre = pg_db.ravel()[_indices]

print(f'Pre-calibration path-gain at RX:  '
      f'mean={np.mean(pg_at_rx_pre):.1f} dB  '
      f'min={np.min(pg_at_rx_pre):.1f} dB  max={np.max(pg_at_rx_pre):.1f} dB')

## CELL 8b · Reference Channel – Ground-Truth OFDM Response

In [ ]:
# ====================================================================
# CELL 8b — CALIBRATION TARGET
# ====================================================================
# If Ofcom measurements are available: use measured RSSI dBm as target
# (power-domain calibration — correct approach for drive-test CSV data)
# Fallback: self-supervised NMSE using compute_paths() (diff-rt demo mode)
# ====================================================================

CALIB_MODE = 'ofcom'   # 'ofcom' = use measured RSSI  |  'self' = self-supervised NMSE

if CALIB_MODE == 'ofcom' and rssi_measured_all is not None:
    # ── Stratified sample of CALIB_N_RX receivers ──────────────────────────
    import math as _math
    _rx_names = [rx.name for rx in receivers]
    _rx_rssi  = {r: rssi_measured_all[i] for i, r in enumerate(_rx_names)
                 if i < len(rssi_measured_all) and np.isfinite(rssi_measured_all[i])}

    # Compute distances for stratified sampling
    _tx_x, _tx_y = gps_to_local(TX_LON, TX_LAT)[:2]
    _dists = {rx.name: float(np.sqrt(
        (_safe(rx.position[0]) - _tx_x)**2 +
        (_safe(rx.position[1]) - _tx_y)**2)) / 1000.0
        for rx in receivers}

    # Stratified: equal-count bins across distance range
    _valid_rx = [rx for rx in receivers if rx.name in _rx_rssi]
    _valid_rx.sort(key=lambda r: _dists[r.name])
    _n_bins  = max(1, CALIB_N_RX // 20)
    _bin_sz  = max(1, len(_valid_rx) // _n_bins)
    _sel_idx = []
    for _b in range(_n_bins):
        _bin = _valid_rx[_b*_bin_sz : (_b+1)*_bin_sz]
        _k   = max(1, CALIB_N_RX // _n_bins)
        _step = max(1, len(_bin) // _k)
        _sel_idx += [receivers.index(r) for r in _bin[::_step]][:_k]
    _sel_idx = sorted(set(_sel_idx))[:CALIB_N_RX]

    calib_receivers  = [receivers[i] for i in _sel_idx]
    calib_rssi_meas  = tf.constant(
        [_rx_rssi[rx.name] for rx in calib_receivers], dtype=tf.float32)

    h_ref_tf = None  # not used in 'ofcom' mode

    print(f'Calibration mode  : Ofcom RSSI  ({len(calib_receivers)} receivers)')
    print(f'RSSI range        : {float(calib_rssi_meas.numpy().min()):.1f} – '
          f'{float(calib_rssi_meas.numpy().max()):.1f} dBm')
    _d_sel = [_dists[rx.name] for rx in calib_receivers]
    print(f'Distance range    : {min(_d_sel):.2f} – {max(_d_sel):.2f} km')

else:
    # ── Self-supervised fallback ────────────────────────────────────────────
    CALIB_MODE = 'self'
    print('Calibration mode  : self-supervised NMSE (no Ofcom data)')
    print(f'Computing reference channel  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

    def compute_h_freq(sc, num_samp=CALIB_NUM_SAMP, depth=CALIB_DEPTH):
        paths = sc.compute_paths(
            max_depth=depth, num_samples=num_samp,
            los=True, reflection=True, scattering=True, diffraction=False)
        if _HAS_OFDM:
            try:
                a, tau = paths.cir()
                h = cir_to_ofdm_channel(FREQUENCIES, a, tau, normalize=False)
                return tf.squeeze(h)
            except Exception as _e:
                print(f'  cir() fallback: {_e}')
        a_t = paths.a
        if isinstance(a_t, tuple): a_t = tf.complex(a_t[0], a_t[1])
        if a_t.shape[0] == 1: a_t = a_t[0]
        return tf.cast(tf.reduce_sum(tf.abs(a_t)**2,
                        axis=list(range(1, len(a_t.shape)))), tf.float32)

    h_ref    = compute_h_freq(scene, num_samp=NUM_SAMPLES_PS, depth=MAX_DEPTH)
    h_ref_np = _to_numpy(h_ref)
    h_ref_tf = tf.constant(h_ref_np,
                            dtype=tf.complex64 if np.iscomplexobj(h_ref_np) else tf.float32)
    calib_receivers = list(receivers)
    calib_rssi_meas = None
    print(f'Reference shape   : {h_ref_np.shape}  dtype={h_ref_np.dtype}')

print(f'\nCalibration receivers : {len(calib_receivers)}')
print(f'Mode                  : {CALIB_MODE}')


## CELL 10 · Differentiable RT – Material Calibration

| Step | What | API |
|------|------|-----|
| 10.1 | Create `RadioMaterial` with `tf.Variable` properties | `RadioMaterial(name, relative_permittivity=tf.Variable(...))` |
| 10.2 | Redirect scene objects to trainable material | `obj.radio_material = name + '_train'` |
| 10.3 | NMSE loss over OFDM channel | `‖ĥ − h_ref‖² / ‖h_ref‖²` |
| 10.4 | Gradients via `tape.watched_variables()` | automatic |
| 10.5 | Clamp to physical range after each step | `check_mat()` |

In [ ]:
orig_params    = {}
original_mats  = {}
trainable_mats = {}
_train_suffix  = '_train'

print('Creating trainable RadioMaterial objects ...')
for mat_name, mat in list(scene.radio_materials.items()):
    if mat_name.endswith(_train_suffix): continue

    _used = False
    try:
        _used = mat.is_used
    except AttributeError:
        _used = any(
            getattr(obj, 'radio_material', None) is not None and
            getattr(getattr(obj, 'radio_material', None), 'name', '') == mat_name
            for obj in scene.objects.values())
    if not _used: continue

    key  = _match_itu(mat_name)
    _itu = _ITU_DB.get(key, _DEFAULT_MAT)
    eps0 = _itu[0]; sig0 = _itu[1]; S0 = _itu[2]
    try:
        v = mat.relative_permittivity
        eps0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    try:
        v = mat.conductivity
        sig0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    for a_ in ('scattering_coefficient','scattering_coeff'):
        if hasattr(mat, a_):
            try: S0 = float(getattr(mat,a_).numpy() if hasattr(getattr(mat,a_),'numpy') else getattr(mat,a_)); break
            except: pass

    orig_params[mat_name] = {'eps_r': eps0, 'sigma': sig0, 'S': S0}

    sn = mat_name.replace('/','_').replace(' ','_').replace('-','_')
    # Log-parameterise conductivity for numerical stability (spans 9 orders of magnitude)
    _log_sig0 = float(np.log(max(sig0, 1e-6)))
    kw = dict(
        relative_permittivity = tf.Variable(eps0,     dtype=tf.float32, name=f'{sn}_eps'),
        conductivity          = tf.Variable(_log_sig0, dtype=tf.float32, name=f'{sn}_log_sig'),
        # NOTE: conductivity variable stores LOG(sigma) — exponentiated when assigned to mat
    )
    try:
        new_mat = RadioMaterial(mat_name+_train_suffix,
                                scattering_coefficient=tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S'),
                                **kw)
    except TypeError:
        new_mat = RadioMaterial(mat_name+_train_suffix, **kw)
        try: new_mat.scattering_coefficient = tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S')
        except: pass

    # Assign exp(log_sigma) as actual conductivity
    try:
        new_mat.conductivity = tf.exp(kw['conductivity'])
    except Exception:
        pass

    scene.add(new_mat)
    original_mats[mat_name]  = mat
    trainable_mats[mat_name] = new_mat
    print(f'  {mat_name:<30} → {mat_name+_train_suffix}')
    print(f'    eps_r={eps0:.3f}  log_sig={_log_sig0:.4g}  S={S0:.2f}')

print()
n_redir = 0
for obj_name, obj in scene.objects.items():
    rm = getattr(obj, 'radio_material', None)
    if rm is None: continue
    orig_name = rm.name if hasattr(rm,'name') else str(rm)
    if orig_name in trainable_mats:
        try:
            obj.radio_material = orig_name + _train_suffix
            n_redir += 1
        except Exception as e:
            print(f'  WARNING [{obj_name}]: {e}')

print(f'Redirected {n_redir} scene objects to trainable materials.')
print(f'Trainable materials: {list(trainable_mats.keys())}')


In [ ]:
# ====================================================================
# LOSS FUNCTIONS — matching CALIB_MODE
# ====================================================================

def smape_power_loss(rssi_sim_dbm, rssi_meas_dbm):
    """
    SMAPE on linear power — official diff-rt-calibration loss (Hoydis et al. 2023).
    More robust than MSE on dBm: scale-invariant, symmetric.
    """
    P_sim  = tf.pow(10.0, (rssi_sim_dbm  - 30.0) / 10.0)   # dBm → Watts
    P_meas = tf.pow(10.0, (rssi_meas_dbm - 30.0) / 10.0)
    P_sim  = tf.cast(P_sim,  tf.float32)
    P_meas = tf.cast(P_meas, tf.float32)
    return tf.reduce_mean(
        tf.abs(P_sim - P_meas) / (P_sim + P_meas + 1e-30))

def mse_dbm_loss(rssi_sim_dbm, rssi_meas_dbm):
    """MSE on dBm — simpler alternative, biased toward strong signals."""
    err = tf.cast(rssi_sim_dbm, tf.float32) - tf.cast(rssi_meas_dbm, tf.float32)
    return tf.reduce_mean(err ** 2)

def nmse_loss(h_hat, h_ref):
    """NMSE — used in self-supervised mode only."""
    h_hat = tf.cast(h_hat, h_ref.dtype)
    err   = tf.reduce_mean(tf.abs(h_hat - h_ref)**2)
    ref   = tf.reduce_mean(tf.abs(h_ref)**2) + 1e-30
    return err / ref

def paths_to_rssi(paths, tx_pwr_dbm, sys_gain_db, eps=1e-30):
    """Extract total received power from paths → RSSI dBm per receiver."""
    a_t = paths.a
    if isinstance(a_t, tuple):
        a_t = tf.complex(a_t[0], a_t[1])
    # Sum power over all paths and antennas: [rx]
    pwr = tf.reduce_sum(tf.abs(a_t)**2, axis=list(range(1, len(a_t.shape))))
    pwr = tf.cast(pwr, tf.float32)
    pwr = tf.squeeze(pwr)
    # Convert to dBm: RSSI = EIRP - PathLoss + SYS_GAIN
    # path_gain = pwr (linear), so PathLoss_linear = 1/path_gain
    rssi_dbm = 10.0 * tf.experimental.numpy.log10(pwr + eps) + 30.0 + tx_pwr_dbm + sys_gain_db
    return rssi_dbm

def check_mat(mat):
    """Clamp material properties to physical range."""
    try:
        v = mat.relative_permittivity
        if hasattr(v, 'assign'):
            v.assign(tf.clip_by_value(v, 1.0, 50.0))
    except Exception: pass
    try:
        v = mat.conductivity
        if hasattr(v, 'assign'):
            # If log-parameterised: keep log_sigma in reasonable range
            v.assign(tf.clip_by_value(v, tf.math.log(1e-6), tf.math.log(1e7)))
    except Exception: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try:
                v = getattr(mat, a_)
                if hasattr(v, 'assign'):
                    v.assign(tf.clip_by_value(v, 1e-3, 1.0 - 1e-3))
                break
            except Exception: pass

print(f'Loss functions ready — active mode: {CALIB_MODE}')
print(f'  Ofcom mode : SMAPE on linear power (Hoydis et al. 2023)')
print(f'  Self mode  : NMSE on OFDM channel')


In [ ]:
# ====================================================================
# CELL 10b — MATERIAL CALIBRATION TRAINING LOOP
# ====================================================================
import random

optimizer = tf.keras.optimizers.Adam(learning_rate=CALIB_LR)

# Collect all trainable variables (eps, log_sig, S for each material)
all_train_vars = []
for mat in trainable_mats.values():
    for attr in ('relative_permittivity', 'conductivity'):
        v = getattr(mat, attr, None)
        if isinstance(v, tf.Variable): all_train_vars.append(v)
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            v = getattr(mat, a_)
            if isinstance(v, tf.Variable): all_train_vars.append(v)
            break

print(f'Trainable variables : {len(all_train_vars)}')
print(f'Materials           : {len(trainable_mats)}')
print(f'Calibration mode    : {CALIB_MODE}')
print(f'Steps               : {CALIB_STEPS}  (batch={CALIB_BATCH if CALIB_MODE=="ofcom" else "all"})')
print(f'Loss function       : {"SMAPE (linear power)" if CALIB_MODE=="ofcom" else "NMSE (OFDM channel)"}')
print('-' * 70)

# ── Helper: set only batch receivers active ───────────────────────────────────
def set_batch_receivers(batch_rxs):
    for nm in list(scene.receivers.keys()):
        scene.remove(nm)
    for rx in batch_rxs:
        scene.add(rx)

history = {'step': [], 'loss_db': [],
           'eps_r': {m: [] for m in trainable_mats},
           'log_sigma': {m: [] for m in trainable_mats},
           'S': {m: [] for m in trainable_mats}}

t0 = time.time()
best_loss = float('inf')
best_params = {}

for step in range(CALIB_STEPS):
    # ── Mini-batch selection ─────────────────────────────────────────────────
    if CALIB_MODE == 'ofcom':
        _idx = random.sample(range(len(calib_receivers)), min(CALIB_BATCH, len(calib_receivers)))
        _idx.sort()
        batch_rx   = [calib_receivers[i] for i in _idx]
        batch_meas = tf.gather(calib_rssi_meas, _idx)
        set_batch_receivers(batch_rx)

    with tf.GradientTape() as tape:
        tape.watch(all_train_vars)

        # Apply exp(log_sigma) to actual material conductivity
        for mat in trainable_mats.values():
            v = getattr(mat, 'conductivity', None)
            if isinstance(v, tf.Variable):
                try:
                    mat.conductivity = tf.exp(tf.clip_by_value(v, -13.8, 16.1))
                except Exception:
                    pass

        _ps_cfg = dict(
            max_depth   = CALIB_DEPTH,
            num_samples = CALIB_NUM_SAMP,
            los         = True,
            diffraction = False,
        )
        # Sionna 2.0 / 0.19 compatible reflection params
        try:
            paths = scene.compute_paths(reflection=True, scattering=True, **_ps_cfg)
        except TypeError:
            paths = scene.compute_paths(
                specular_reflection=True, diffuse_reflection=True,
                refraction=True, **_ps_cfg)

        if CALIB_MODE == 'ofcom':
            rssi_sim = paths_to_rssi(paths, TX_CONDUCTED_DBM, RX_EXTRA_GAIN_DB)
            # Pad/trim to batch size
            n = min(rssi_sim.shape[0], batch_meas.shape[0])
            loss = smape_power_loss(rssi_sim[:n], batch_meas[:n])
        else:
            h_hat  = compute_h_freq(scene, CALIB_NUM_SAMP, CALIB_DEPTH)
            h_hat  = tf.cast(h_hat, h_ref_tf.dtype)
            n      = min(h_hat.shape[0], h_ref_tf.shape[0])
            loss   = nmse_loss(h_hat[:n], h_ref_tf[:n])

    grads = tape.gradient(loss, all_train_vars,
                          unconnected_gradients=tf.UnconnectedGradients.ZERO)
    optimizer.apply_gradients(zip(grads, all_train_vars))

    for mat in trainable_mats.values():
        check_mat(mat)

    loss_f  = float(loss.numpy())
    loss_db = 10 * np.log10(loss_f + 1e-30) if CALIB_MODE == 'self' else loss_f * 100
    if loss_f < best_loss:
        best_loss = loss_f
        best_params = {mn: {
            'eps_r': _safe(mat.relative_permittivity),
            'log_sigma': _safe(mat.conductivity),
        } for mn, mat in trainable_mats.items()}

    history['step'].append(step)
    history['loss_db'].append(loss_db)
    for mn, mat in trainable_mats.items():
        try: history['eps_r'][mn].append(_safe(mat.relative_permittivity))
        except: history['eps_r'][mn].append(float('nan'))
        try: history['log_sigma'][mn].append(_safe(mat.conductivity))
        except: history['log_sigma'][mn].append(float('nan'))
        for a_ in ('scattering_coefficient', 'scattering_coeff'):
            if hasattr(mat, a_):
                try: history['S'][mn].append(_safe(getattr(mat, a_))); break
                except: pass
        else:
            history['S'][mn].append(float('nan'))

    if step % 50 == 0 or step == CALIB_STEPS - 1:
        label = 'SMAPE×100' if CALIB_MODE == 'ofcom' else 'NMSE(dB)'
        n_zero = sum(1 for g in grads if g is None or float(tf.reduce_sum(tf.abs(g))) == 0)
        print(f'  step {step:4d}  {label}={loss_db:+7.2f}  '
              f'zero_grads={n_zero}/{len(grads)}  t={time.time()-t0:.0f}s')

# Restore all receivers
set_batch_receivers(calib_receivers)

print('-' * 70)
print(f'Done in {time.time()-t0:.1f}s  |  Best loss: {best_loss:.6f}')


In [ ]:
n_mats   = len(trainable_mats)
colors   = plt.cm.tab10(np.linspace(0, 1, max(n_mats, 1)))
mat_list = list(trainable_mats.keys())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['step'], history['nmse_db'], 'k-', lw=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('NMSE (dB)')
axes[0].set_title('Calibration Loss (NMSE)'); axes[0].grid(True, alpha=0.4)

for i, mn in enumerate(mat_list):
    axes[1].plot(history['step'], history['eps_r'][mn],
                 label=mn.replace('_train',''), color=colors[i])
axes[1].set_xlabel('Step'); axes[1].set_ylabel('ε_r')
axes[1].set_title('Relative Permittivity'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.4)

for i, mn in enumerate(mat_list):
    axes[2].plot(history['step'], history['sigma'][mn],
                 label=mn.replace('_train',''), color=colors[i])
axes[2].set_xlabel('Step'); axes[2].set_ylabel('σ (S/m)')
axes[2].set_title('Conductivity'); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'calibration_convergence.png'), dpi=150)
plt.show()

In [ ]:
print(f'{"Material":<30} {"Param":<8} {"Initial":>10} {"Calibrated":>12} {"Delta":>8}')
print('=' * 75)

calib_results = {}
for mn, mat in trainable_mats.items():
    orig = orig_params.get(mn, {})
    row  = {}
    for pname, attr_candidates, key in [
        ('eps_r', ['relative_permittivity'], 'eps_r'),
        ('sigma', ['conductivity'],           'sigma'),
        ('S',     ['scattering_coefficient', 'scattering_coeff'], 'S'),
    ]:
        init_val = orig.get(key, float('nan'))
        cal_val  = float('nan')
        for a_ in attr_candidates:
            if hasattr(mat, a_):
                try: cal_val = _safe(getattr(mat, a_)); break
                except: pass
        delta = cal_val - init_val if not np.isnan(init_val) else float('nan')
        print(f'  {mn.replace(_train_suffix,""):<28} {pname:<8} '
              f'{init_val:>10.4f} {cal_val:>12.4f} {delta:>+8.4f}')
        row[pname] = {'initial': init_val, 'calibrated': cal_val}
    calib_results[mn] = row

out_json = os.path.join(OUTPUT_DIR, 'calibration_results.json')
with open(out_json, 'w') as f:
    json.dump(calib_results, f, indent=2)
print(f'\nCalibration JSON saved to {out_json}')

## CELL 11 · TX Orientation Optimization

| Parameter | Value |
|-----------|-------|
| Optimizer | **RMSprop** (diff-rt choice) |
| Loss | −E[log₂(1 + SNR × path_gain)] |
| Variable | `tx.orientation` as `tf.Variable` |
| Steps | `ORI_STEPS` |

In [ ]:
tx_name = list(scene.transmitters.keys())[0]
tx      = scene.transmitters[tx_name]

_ori_init = [0.0, 0.0, 0.0]
try: _ori_init = [_safe(tx.orientation[i]) for i in range(3)]
except: pass

tx.orientation = tf.Variable(_ori_init, dtype=tf.float32, name='tx_orientation')
print(f'TX "{tx_name}"  orientation = {_ori_init}  → tf.Variable')

def cm_capacity_loss(sc, cell_size=10.0, n_samp=ORI_NUM_SAMP):
    """Loss = −E[log₂(1 + SNR_scale × path_gain)]  (diff-rt Learning_Orientation)."""
    try:
        cm = sc.coverage_map(
            cm_cell_size        = cell_size,
            max_depth           = 3,
            num_samples         = n_samp,
            los                 = True,
            specular_reflection = True,
            diffuse_reflection  = False,
            refraction          = False,
            diffraction         = False,
        )
        pg = None
        for attr in ('path_gain', 'as_tensor'):
            if hasattr(cm, attr):
                val = getattr(cm, attr)
                pg  = val() if callable(val) else val
                break
        if pg is None: raise AttributeError('no path_gain')
        pg_flat  = tf.reshape(tf.cast(pg[0], tf.float32), [-1])
        capacity = tf.reduce_mean(
            tf.math.log(1.0 + SNR_SCALE * pg_flat) / tf.math.log(2.0))
        return -capacity, cm
    except Exception as e:
        print(f'  cm_capacity_loss error: {e}')
        return tf.constant(0.0), None

print(f'SNR_SCALE = {SNR_SCALE:.2e}')

In [ ]:
ori_optimizer = tf.keras.optimizers.RMSprop(learning_rate=ORI_LR)
ori_history   = {'step': [], 'rate_bit': [], 'orientation': []}

print(f'TX orientation optimization – {ORI_STEPS} steps  RMSprop LR={ORI_LR}')
print('-' * 60)

cm_before_np = None
t0 = time.time()
for step in range(ORI_STEPS):
    with tf.GradientTape() as tape:
        loss_val, cm_opt = cm_capacity_loss(scene, cell_size=10.0, n_samp=ORI_NUM_SAMP)

    if step == 0 and cm_opt is not None:
        cm_before_np = _cm_to_numpy(cm_opt)

    grads    = tape.gradient(loss_val, tape.watched_variables())
    valid_gv = [(g, v) for g, v in zip(grads, tape.watched_variables()) if g is not None]
    if valid_gv: ori_optimizer.apply_gradients(valid_gv)

    rate = float(-loss_val.numpy())
    ori  = list(tx.orientation.numpy())
    ori_history['step'].append(step)
    ori_history['rate_bit'].append(rate)
    ori_history['orientation'].append(ori)
    print(f'  step {step:3d}  rate={rate:.4f} bit  '
          f'ori=[{", ".join(f"{o:.3f}" for o in ori)}]  t={time.time()-t0:.0f}s', end='\r')

print()
print('-' * 60)
ori_final = list(tx.orientation.numpy())
print(f'Initial orientation  : {_ori_init}')
print(f'Optimized orientation: {[round(o, 4) for o in ori_final]}')
rate_initial = ori_history['rate_bit'][0]  if ori_history['rate_bit'] else 0
rate_final   = ori_history['rate_bit'][-1] if ori_history['rate_bit'] else 0
print(f'Rate improvement     : {rate_initial:.4f} → {rate_final:.4f} bit  (+{rate_final-rate_initial:.4f})')

## CELL 12 · Post-Calibration Analysis

In [ ]:
print('Post-calibration path computation ...')
print(f'  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

paths_cal = scene.compute_paths(
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_PS,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = False,
)
print('Done.')

a_np   = _to_numpy(paths_cal.a)
tau_np = _to_numpy(paths_cal.tau)
if a_np.ndim == 6: a_np = a_np[0]

n_rx    = a_np.shape[0]
power   = np.sum(np.abs(a_np)**2, axis=tuple(range(1, a_np.ndim)))
pg_cal  = power
pg_cal_db = 10 * np.log10(pg_cal + 1e-30)

print(f'Post-calibration path gain at {n_rx} receivers:')
print(f'  mean={pg_cal_db.mean():.1f} dB  min={pg_cal_db.min():.1f} dB  max={pg_cal_db.max():.1f} dB')

In [ ]:
records = []
for i, rx in enumerate(receivers[:n_rx]):
    x   = _safe(rx.position[0])
    y   = _safe(rx.position[1])
    z   = _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    pg_pre  = float(pg_at_rx_pre[i]) if i < len(pg_at_rx_pre) else float('nan')
    pg_post = float(pg_cal_db[i])    if i < len(pg_cal_db)    else float('nan')
    records.append({
        'receiver'    : rx.name,
        'lon'         : round(lon, 6),
        'lat'         : round(lat, 6),
        'x_m'         : round(x,   2),
        'y_m'         : round(y,   2),
        'z_m'         : round(z,   3),
        'pg_pre_db'   : round(pg_pre,  2),
        'pg_post_db'  : round(pg_post, 2),
        'delta_pg_db' : round(pg_post - pg_pre, 2) if not np.isnan(pg_pre) else float('nan'),
    })

df_out = pd.DataFrame(records)
out_csv = os.path.join(OUTPUT_DIR, 'receiver_results_calibrated.csv')
df_out.to_csv(out_csv, index=False)
print(f'Saved {len(df_out)} receivers to {out_csv}')
print(df_out.head(10).to_string(index=False))

In [ ]:
print('Computing final calibrated coverage map ...')
cm_final    = scene.coverage_map(
    cm_cell_size        = GRID_SIZE_M,
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_CM,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = False,
)
cm_final_np    = _cm_to_numpy(cm_final)
pg_pre_db_2d   = 10 * np.log10(cm_pre_np[0]   + 1e-30)
pg_final_db_2d = 10 * np.log10(cm_final_np[0] + 1e-30)

vmin = min(np.nanpercentile(pg_pre_db_2d, 5),  np.nanpercentile(pg_final_db_2d, 5))
vmax = max(np.nanpercentile(pg_pre_db_2d, 99), np.nanpercentile(pg_final_db_2d, 99))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, data, title in [
    (axes[0], pg_pre_db_2d,   'Before Calibration (ITU defaults)'),
    (axes[1], pg_final_db_2d, 'After Calibration (diff-rt)'),
    (axes[2], pg_final_db_2d - pg_pre_db_2d, 'Δ Path Gain (After − Before)'),
]:
    if 'Δ' in title:
        im = ax.imshow(data, origin='lower', cmap='RdYlGn', vmin=-10, vmax=10)
    else:
        im = ax.imshow(data, origin='lower', cmap='jet', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Path Gain (dB)')
    ax.set_title(title); ax.set_xlabel('X cells'); ax.set_ylabel('Y cells')

plt.suptitle('Coverage Map: Before vs After Differentiable RT Calibration', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nAll results saved to:', OUTPUT_DIR)